# 10 · Optimización de inferencia y pipelines 

La oferta: *"optimización de modelos y pipelines para conseguir tiempos de procesamiento reducidos"*,
*"acelerar procesos de inferencia"*, y valoran *CUDA, TensorRT, ONNX, DeepStream, GStreamer, FFmpeg*.

1. **Medir bien** (warm-up, sincronización, por etapas) ✍️
2. Latencia vs throughput: batching
3. Hacer menos trabajo: resolución, modelo, frame skipping
4. Precisión numérica: FP16, INT8, cuantización
5. Exportación: ONNX, TensorRT, CoreML — y validar que no se pierde precisión
6. El pipeline completo: decodificación, hilos, colas ✍️
7. CUDA y GPU en lo que hay que saber explicar

En el Mac no hay CUDA/TensorRT: medimos con PyTorch-MPS, ONNX Runtime (CPU) y CoreML. Al final tienes el bloque
para repetirlo con TensorRT en Google Colab (GPU T4 gratuita).

In [ ]:
# ⚙️ SETUP · ejecuta esta celda primero. Hace el notebook independiente: funciona en local (dentro del repo)
# y en Google Colab (instala lo que falte y descarga los datos a ./data).
import sys, subprocess, importlib.util, urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
PIP = {"ultralytics": "ultralytics", "onnx": "onnx", "onnxruntime": "onnxruntime", "onnxslim": "onnxslim"}   # paquete pip → módulo que importa
missing = [p for p, m in PIP.items() if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# Raíz del repo: la carpeta que contiene vision_utils/ (subiendo desde aquí, o el clon de Colab)
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents, Path.cwd() / "vision-bootcamp"]
             if (p / "vision_utils").is_dir()), None)
ROOT = ROOT or Path.cwd()   # fuera del repo (Colab): los datos se guardan en ./data
DATA = ROOT / "data"
DATA.mkdir(exist_ok=True)
if not (DATA / "people.mp4").exists():   # vídeo de ejemplo (cámara fija, ~14 s, 1920×1080)
    urllib.request.urlretrieve("https://media.roboflow.com/supervision/video-examples/people-walking.mp4",
                               DATA / "people.mp4")
# Los pesos YOLO (yolo11n.pt…) no hace falta descargarlos: Ultralytics los baja solo la primera vez que se usan.
print("ROOT:", ROOT, "| DATA:", DATA, "| Colab:", IN_COLAB)

In [ ]:
import time, threading, queue
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import torch
import matplotlib.pyplot as plt
from ultralytics import YOLO

VIDEO = DATA / "people.mp4"
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
cap = cv2.VideoCapture(str(VIDEO)); ok, frame = cap.read(); cap.release()

## 1. Medir bien

Tres errores clásicos al medir velocidad:
1. **No hacer warm-up**: la primera inferencia incluye inicialización (carga de kernels, autotuning de cuDNN,
   compilación de Metal) → siempre descarta las primeras iteraciones.
2. **No sincronizar la GPU**: las operaciones en GPU son **asíncronas**; `model(x)` devuelve antes de terminar.
   Si mides sin `torch.cuda.synchronize()` (o `torch.mps.synchronize()`), mides solo el lanzamiento.
3. **Medir solo el modelo**: en un pipeline real el tiempo se reparte entre decodificar, preprocesar, inferir,
   postprocesar (NMS), tracking, dibujar/escribir. **Mide cada etapa**.

✍️ **EJERCICIO**: escribe `benchmark(fn, n=50, warmup=10)` que devuelva la mediana y p95 en ms, sincronizando
el dispositivo.

In [ ]:
def sync():
    if DEVICE == "cuda": torch.cuda.synchronize()
    elif DEVICE == "mps": torch.mps.synchronize()

def benchmark(fn, n=50, warmup=10):
    # TU CÓDIGO AQUÍ → dict(median_ms=..., p95_ms=...)
    raise NotImplementedError

In [ ]:
# SOLUCIÓN
def benchmark(fn, n=50, warmup=10):
    for _ in range(warmup):
        fn()
    sync()
    ts = []
    for _ in range(n):
        t0 = time.perf_counter()
        fn()
        sync()
        ts.append((time.perf_counter() - t0) * 1000)
    return {"median_ms": float(np.median(ts)), "p95_ms": float(np.percentile(ts, 95))}

In [ ]:
model = YOLO(str(DATA / "yolo11n.pt"))
net = model.model.to(DEVICE).eval()
x = torch.rand(1, 3, 640, 640, device=DEVICE)

def no_sync_time():
    t0 = time.perf_counter()
    with torch.inference_mode(): net(x)
    return (time.perf_counter() - t0) * 1000

with torch.inference_mode():
    for _ in range(5): net(x)
sync()
print(f"sin sincronizar: {np.median([no_sync_time() for _ in range(20)]):.2f} ms  ← engañoso (puede salir demasiado rápido)")
with torch.inference_mode():
    print("con sincronización:", benchmark(lambda: net(x)))

In [ ]:
# Tiempo por etapas con el pipeline de Ultralytics (r.speed ya separa pre/inferencia/post)
cap = cv2.VideoCapture(str(VIDEO))
stages = []
for i in range(60):
    t0 = time.perf_counter(); ok, f = cap.read(); t_dec = (time.perf_counter() - t0) * 1000
    r = model(f, device=DEVICE, classes=[0], verbose=False)[0]
    t0 = time.perf_counter(); r.plot(); t_draw = (time.perf_counter() - t0) * 1000
    if i >= 10:
        stages.append({"decode": t_dec, **r.speed, "draw": t_draw})
cap.release()
st = pd.DataFrame(stages).median()
st.plot.barh(title="ms por etapa (mediana)"); plt.show()
print(st.round(1), f"\ntotal ≈ {st.sum():.1f} ms → {1000 / st.sum():.0f} FPS")

## 2. Latencia vs throughput: batching

- **Latencia**: tiempo de un frame (importa en directo).
- **Throughput**: frames por segundo procesados (importa en offline o con muchas cámaras).
- La GPU es masivamente paralela; con batch=1 un modelo pequeño no la llena. **Agrupar frames** (de varias
  cámaras o frames consecutivos en offline) sube el throughput, a costa de latencia y memoria.

In [ ]:
rows = []
for bs in [1, 2, 4, 8, 16]:
    xb = torch.rand(bs, 3, 640, 640, device=DEVICE)
    with torch.inference_mode():
        r = benchmark(lambda: net(xb), n=15, warmup=3)
    rows.append({"batch": bs, "ms/batch": r["median_ms"], "ms/imagen": r["median_ms"] / bs, "img/s": 1000 * bs / r["median_ms"]})
pd.DataFrame(rows).round(1)

## 3. Hacer menos trabajo (lo primero que hay que probar)

- **Modelo más pequeño** (n/s) y **menor `imgsz`** (coste ∝ píxeles: 640→480 ≈ 44% menos). Medir el impacto en
  mAP y en la métrica final (HOTA).
- **Recortar la región de interés** (solo el campo, no las gradas) o tiling solo donde hace falta.
- **Frame skipping**: detectar cada N frames y en los intermedios solo `predict` del Kalman (o un tracker ligero
  tipo flujo óptico). A 50 FPS, detectar a 25 suele bastar.
- **Filtrar clases** y subir `conf` mínimo → menos cajas para NMS y tracker.
- Evitar trabajo de Python por objeto: vectorizar, no dibujar si no hace falta, no convertir a CPU antes de tiempo.

In [ ]:
rows = []
for name in ["yolo11n.pt", "yolo11s.pt"]:
    m = YOLO(str(DATA / name))
    for sz in [480, 640, 960]:
        r = benchmark(lambda: m(frame, imgsz=sz, classes=[0], device=DEVICE, verbose=False), n=15, warmup=3)
        n_det = len(m(frame, imgsz=sz, classes=[0], device=DEVICE, verbose=False)[0].boxes)
        rows.append({"modelo": name, "imgsz": sz, "ms": round(r["median_ms"], 1), "personas": n_det})
pd.DataFrame(rows).pivot(index="imgsz", columns="modelo", values=["ms", "personas"])

## 4. Precisión numérica

| Formato | Bits | Ganancia típica en GPU NVIDIA | Pérdida de precisión | Notas |
|---|---|---|---|---|
| FP32 | 32 | 1× | — | entrenamiento por defecto |
| **FP16** / BF16 | 16 | ~2× (Tensor Cores) | casi nula en detección | `half=True`; BF16 = mismo rango que FP32, menos mantisa |
| **INT8** | 8 | ~3–4× vs FP32 | pequeña si se calibra bien | necesita **calibración** con imágenes representativas |

**Cuantización**:
- **PTQ** (post-training quantization): convertir un modelo entrenado; para INT8 se pasan ~500–1000 imágenes de
  calibración para estimar los rangos de activaciones (escala y *zero-point*). Rápido; puede perder algo.
- **QAT** (quantization-aware training): simular la cuantización durante el entrenamiento → recupera precisión.
- Riesgo: capas sensibles (la cabeza de detección, la regresión de cajas) → se dejan en FP16 (*mixed precision*).
- **Siempre validar** después: mAP del modelo exportado vs original, y la métrica final del sistema.

Otras técnicas de compresión: **pruning** (eliminar canales/pesos), **knowledge distillation** (entrenar un
modelo pequeño imitando a uno grande), y elegir arquitecturas eficientes.

## 5. Exportación: ONNX, TensorRT, CoreML

- **ONNX**: formato abierto del grafo del modelo (operadores estándar + pesos). Independiente del framework →
  se ejecuta con **ONNX Runtime** (CPU, CUDA, TensorRT, CoreML, OpenVINO como *execution providers*) y es la
  puerta de entrada a TensorRT. Conceptos: `opset`, **ejes dinámicos** (batch/tamaño variable), simplificación
  del grafo (onnxslim), ops no soportadas.
- **TensorRT** (NVIDIA): compila el grafo a un **engine** optimizado para **esa GPU y esa versión de TensorRT**:
  **fusión de capas** (conv+BN+activación en un kernel), **autotuning** de kernels, FP16/INT8, gestión de memoria.
  Típicamente 2–5× sobre PyTorch. El engine no es portable entre GPUs/versiones → se construye en la máquina de
  destino (o en una idéntica). Ultralytics: `model.export(format="engine", half=True)` o `int8=True, data=...`.
- **CoreML** (Apple, usa la GPU y el Neural Engine), **OpenVINO** (CPU/iGPU Intel), **TFLite** (móvil).
- `torch.compile` (PyTorch 2): compilación JIT del modelo en PyTorch sin salir de él.

In [ ]:
onnx_path = model.export(format="onnx", imgsz=640, simplify=True, dynamic=False, verbose=False)
print("exportado:", onnx_path)

import onnxruntime as ort
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
inp = sess.get_inputs()[0]
print("entrada ONNX:", inp.name, inp.shape, "| salida:", sess.get_outputs()[0].shape)

x_np = np.random.rand(1, 3, 640, 640).astype(np.float32)
net_cpu = YOLO(str(DATA / "yolo11n.pt")).model.float().eval()
with torch.inference_mode():
    y_pt = net_cpu(torch.from_numpy(x_np))
y_pt = (y_pt[0] if isinstance(y_pt, (list, tuple)) else y_pt).numpy()
y_ort = sess.run(None, {inp.name: x_np})[0]
print(f"diferencia máx PyTorch vs ONNX: {np.abs(y_pt - y_ort).max():.2e} (las cajas están en píxeles de 0–640 → despreciable)")
print("← validar SIEMPRE que la exportación es fiel (y después con mAP, abajo)")

In [ ]:
res = {}
with torch.inference_mode():
    res["PyTorch CPU"] = benchmark(lambda: net_cpu(torch.from_numpy(x_np)), n=20, warmup=3)
    res[f"PyTorch {DEVICE}"] = benchmark(lambda: net(x), n=20, warmup=3)
res["ONNX Runtime CPU"] = benchmark(lambda: sess.run(None, {inp.name: x_np}), n=20, warmup=3)
try:
    coreml_path = model.export(format="coreml", imgsz=640, half=True, nms=False, verbose=False)
    cm = YOLO(coreml_path, task="detect")
    res["CoreML FP16 (pipeline completo)"] = benchmark(lambda: cm(frame, verbose=False), n=20, warmup=3)
except Exception as e:
    print("CoreML no disponible:", str(e)[:100])
pd.DataFrame(res).T.round(1)

In [ ]:
# Validar precisión del modelo exportado con métricas, no solo con la diferencia de tensores
m_pt = YOLO(str(DATA / "yolo11n.pt")).val(data="coco8.yaml", imgsz=640, device="cpu", verbose=False, plots=False)
m_onnx = YOLO(onnx_path, task="detect").val(data="coco8.yaml", imgsz=640, device="cpu", verbose=False, plots=False)
print(f"mAP50-95  PyTorch: {m_pt.box.map:.4f} | ONNX: {m_onnx.box.map:.4f}")

### TensorRT en Google Colab (opcional)
Abre un notebook en Colab → *Entorno de ejecución → Cambiar tipo → GPU T4* y ejecuta:
```python
!pip -q install ultralytics tensorrt
from ultralytics import YOLO
import time, torch
m = YOLO("yolo11n.pt")
m.export(format="engine", half=True, imgsz=640)        # crea yolo11n.engine (tarda 1–3 min)
trt = YOLO("yolo11n.engine")
img = "https://ultralytics.com/images/bus.jpg"
for name, mm in [("PyTorch FP32", m), ("TensorRT FP16", trt)]:
    for _ in range(10): mm(img, verbose=False)
    t = time.perf_counter()
    for _ in range(100): r = mm(img, verbose=False)
    print(name, r[0].speed)
```
Verás la inferencia bajar de varios ms a ~1–2 ms; y entonces el cuello de botella pasa a ser el pre/postproceso
en CPU → por eso DeepStream lo hace todo en GPU.

## 6. El pipeline completo

Un pipeline secuencial `leer → inferir → trackear → escribir` deja recursos ociosos: mientras la GPU infiere, la
CPU no decodifica. **Productor–consumidor con colas**: un hilo decodifica y llena una cola acotada; otro infiere;
otro escribe. Las etapas se solapan → throughput ≈ el de la etapa más lenta, no la suma.

(En Python los hilos sirven aquí porque OpenCV y PyTorch liberan el GIL durante su trabajo pesado; para CPU
puro en Python habría que usar procesos.)

✍️ **EJERCICIO**: `ThreadedReader` que decodifique en un hilo y entregue frames por una `queue.Queue(maxsize)`.
Compara FPS secuencial vs con hilo.

In [ ]:
class ThreadedReader:
    def __init__(self, path, maxsize=32):
        # TU CÓDIGO AQUÍ
        raise NotImplementedError

    def __iter__(self):
        raise NotImplementedError

In [ ]:
# SOLUCIÓN
class ThreadedReader:
    _END = object()

    def __init__(self, path, maxsize=32):
        self.cap = cv2.VideoCapture(str(path))
        self.q = queue.Queue(maxsize=maxsize)       # acotada → si el consumidor es lento, el productor espera
        self.t = threading.Thread(target=self._run, daemon=True)
        self.t.start()

    def _run(self):
        while True:
            ok, f = self.cap.read()
            if not ok:
                break
            self.q.put(f)
        self.q.put(self._END)
        self.cap.release()

    def __iter__(self):
        while (f := self.q.get()) is not self._END:
            yield f

In [ ]:
def run(frames_iter, n=200):
    t0, k = time.perf_counter(), 0
    for f in frames_iter:
        model(f, device=DEVICE, classes=[0], verbose=False)
        k += 1
        if k >= n: break
    return k / (time.perf_counter() - t0)

def sequential(path):
    cap = cv2.VideoCapture(str(path))
    while True:
        ok, f = cap.read()
        if not ok: break
        yield f
    cap.release()

model(frame, device=DEVICE, verbose=False)                   # warm-up
print(f"secuencial: {run(sequential(VIDEO)):.1f} FPS | lector en hilo: {run(ThreadedReader(VIDEO)):.1f} FPS")

### Pipelines de producción con GPU NVIDIA
```
RTSP/fichero ─► NVDEC (decodificación en GPU) ─► redimensionado/normalización en GPU ─► batch multi-cámara
            ─► TensorRT ─► NMS en GPU ─► tracker ─► metadatos (Kafka / JSON) ─► NVENC (si hay que emitir vídeo)
```
- **Mantener los frames en GPU** de principio a fin: cada copia CPU↔GPU de un 1080p cuesta ~1 ms y satura el bus PCIe.
- **DeepStream** = GStreamer + plugins NVIDIA: `nvstreammux` (batch de N cámaras), `nvinfer` (TensorRT),
  `nvtracker` (NvDCF, IOU, DeepSORT), `nvdsosd` (dibujar), salida de metadatos. Procesa decenas de cámaras por GPU.
- **GStreamer** a mano: `rtspsrc ! rtph264depay ! h264parse ! nvv4l2decoder ! nvvideoconvert ! appsink`.
- **FFmpeg**: `-hwaccel cuda -hwaccel_output_format cuda` para decodificar en GPU; PyAV / torchaudio-StreamReader /
  NVIDIA VPF / DALI para hacerlo desde Python.
- Escalado horizontal: un proceso por GPU / por grupo de cámaras, Triton Inference Server para servir modelos
  con *dynamic batching*.

## 7. CUDA y GPU: lo que hay que saber explicar

- **Modelo de ejecución**: un *kernel* se lanza sobre una rejilla de **bloques** de **hilos**; los hilos se
  ejecutan en grupos de 32 (**warps**) en los SMs. Miles de hilos en paralelo → ideal para operaciones por píxel
  o por elemento.
- **Jerarquía de memoria**: registros → memoria compartida (por bloque, muy rápida) → L2 → **memoria global
  (VRAM)** → memoria del host (CPU) vía PCIe (lenta). Muchas operaciones están **limitadas por memoria**, no por
  cálculo → fusionar operaciones (TensorRT, `torch.compile`) evita idas y vueltas a VRAM.
- **Asincronía y streams**: los kernels se encolan; varios **CUDA streams** permiten solapar copia y cómputo.
  **Pinned memory** (`pin_memory=True`) acelera copias host→device y permite `non_blocking=True`.
- **Tensor Cores**: unidades para multiplicación de matrices en FP16/BF16/INT8 → por eso FP16 da ~2×.
- Ajustes de PyTorch: `torch.backends.cudnn.benchmark = True` (tamaño de entrada fijo), `channels_last`,
  `torch.compile`, AMP, CUDA Graphs (reduce el coste de lanzar kernels en modelos pequeños).
- **Herramientas**: `nvidia-smi` (uso, memoria), **Nsight Systems** (línea temporal CPU/GPU: ver huecos),
  PyTorch Profiler.
- Un kernel CUDA mínimo (C++), por si sale "conocimientos básicos de C++":
```cpp
__global__ void add(const float* a, const float* b, float* c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] + b[i];
}
// add<<<(n + 255) / 256, 256>>>(d_a, d_b, d_c, n);
```